# 03 OLSE reliability and CFA measurement models

Purpose: reproduce RQ2 OLSE item diagnostics, reliability estimates, ordinal CFA models, measurement comparability checks, and SEM-ready OLSE scoring.

Notebook-generated outputs are written to `outputs_notebooks/olse_measurement/`, with SEM-ready processed data in `data/processed_notebooks/`. Run `01_prepare_data.ipynb` first, or run the full pipeline notebook.

In [ ]:
# Notebook setup: make execution robust from either repo root or notebooks/
find_project_root <- function(start = getwd()) {
  current <- normalizePath(start, winslash = "/", mustWork = FALSE)
  repeat {
    has_markers <- file.exists(file.path(current, "data")) &&
      file.exists(file.path(current, "scripts")) &&
      file.exists(file.path(current, "R"))
    if (has_markers) return(current)
    parent <- dirname(current)
    if (identical(parent, current)) stop("Could not find repository root.", call. = FALSE)
    current <- parent
  }
}

PROJECT_ROOT <- find_project_root(getwd())
setwd(PROJECT_ROOT)
message("Project root: ", PROJECT_ROOT)

# Reference locations used by the command-line R scripts.
SCRIPT_DATA_PROCESSED <- file.path(PROJECT_ROOT, "data", "processed")
SCRIPT_OUTPUT_ROOT <- file.path(PROJECT_ROOT, "outputs")
SCRIPT_FIGURES_DIR <- file.path(PROJECT_ROOT, "figures")

# Notebook-generated artifacts are intentionally written separately so reviewers
# can compare notebook outputs against the original script outputs.
NOTEBOOK_DATA_PROCESSED <- file.path(PROJECT_ROOT, "data", "processed_notebooks")
NOTEBOOK_OUTPUT_ROOT <- file.path(PROJECT_ROOT, "outputs_notebooks")
NOTEBOOK_FIGURES_DIR <- file.path(PROJECT_ROOT, "figures_notebooks")
COMPARISON_ROOT <- file.path(PROJECT_ROOT, "outputs_comparison")

Sys.setenv(
  LENS_DATA_PROCESSED_DIR = NOTEBOOK_DATA_PROCESSED,
  LENS_OUTPUT_DIR = NOTEBOOK_OUTPUT_ROOT,
  LENS_FIGURES_DIR = NOTEBOOK_FIGURES_DIR
)

source(file.path("R", "00_packages.R"))
source(file.path("R", "00_paths.R"))
source(file.path("R", "utils_io.R"))

preview_csv <- function(path, n = 8) {
  if (!file.exists(path)) stop("Missing expected file: ", path, call. = FALSE)
  readr::read_csv(path, show_col_types = FALSE) |>
    dplyr::slice_head(n = n)
}

assert_files_exist <- function(paths) {
  missing <- paths[!file.exists(paths)]
  if (length(missing) > 0) {
    stop("Missing expected output file(s):\n", paste(missing, collapse = "\n"), call. = FALSE)
  }
  tibble::tibble(file = paths, size_bytes = file.info(paths)$size)
}

list_relative_files <- function(base_dir, pattern = "\\.(csv|html|xlsx|png)$") {
  if (!dir.exists(base_dir)) return(character())
  files <- list.files(base_dir, pattern = pattern, recursive = TRUE, full.names = TRUE)
  sort(sub(paste0("^", normalizePath(base_dir, winslash = "/", mustWork = FALSE), "/?"), "", normalizePath(files, winslash = "/", mustWork = FALSE)))
}

md5_or_na <- function(path) {
  if (!file.exists(path)) return(NA_character_)
  unname(as.character(tools::md5sum(path)))
}

compare_file_sets <- function(script_base, notebook_base, group, pattern = "\\.(csv|html|xlsx|png)$") {
  script_rel <- list_relative_files(script_base, pattern)
  notebook_rel <- list_relative_files(notebook_base, pattern)
  rel_paths <- sort(unique(c(script_rel, notebook_rel)))
  if (length(rel_paths) == 0) {
    return(tibble::tibble(
      group = character(), relative_path = character(), script_path = character(), notebook_path = character(),
      script_exists = logical(), notebook_exists = logical(), script_md5 = character(), notebook_md5 = character(), status = character()
    ))
  }
  out <- tibble::tibble(
    group = group,
    relative_path = rel_paths,
    script_path = file.path(script_base, rel_paths),
    notebook_path = file.path(notebook_base, rel_paths)
  ) |>
    dplyr::mutate(
      script_exists = file.exists(script_path),
      notebook_exists = file.exists(notebook_path),
      script_md5 = vapply(script_path, md5_or_na, character(1)),
      notebook_md5 = vapply(notebook_path, md5_or_na, character(1)),
      status = dplyr::case_when(
        !script_exists ~ "missing_script_reference",
        !notebook_exists ~ "missing_notebook_output",
        script_md5 == notebook_md5 ~ "match",
        TRUE ~ "different"
      )
    )
  out
}

write_comparison_manifest <- function(filename = "notebook_vs_script_manifest.csv") {
  dir.create(COMPARISON_ROOT, recursive = TRUE, showWarnings = FALSE)
  manifest <- dplyr::bind_rows(
    compare_file_sets(SCRIPT_DATA_PROCESSED, DATA_PROCESSED, "processed_data", "\\.csv$"),
    compare_file_sets(SCRIPT_OUTPUT_ROOT, OUTPUT_ROOT, "analysis_outputs", "\\.(csv|html|xlsx)$"),
    compare_file_sets(SCRIPT_FIGURES_DIR, OUT_FIGURES, "figures", "\\.(csv|png)$")
  )
  out_path <- file.path(COMPARISON_ROOT, filename)
  readr::write_csv(manifest, out_path, na = "")
  message("Wrote comparison manifest: ", out_path)
  manifest
}

message("Notebook processed data directory: ", DATA_PROCESSED)
message("Notebook output directory: ", OUTPUT_ROOT)
message("Notebook figures directory: ", OUT_FIGURES)
message("Comparison manifests directory: ", COMPARISON_ROOT)

## Check required notebook processed input

In [ ]:
assert_files_exist(c(file.path(DATA_PROCESSED, "student_term_fe_analysis.csv")))

## Check additional CFA packages

In [ ]:
extra_packages <- c("lavaan", "semTools", "psych")
missing_extra <- extra_packages[!vapply(extra_packages, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing_extra) > 0) {
  stop("Install missing CFA package(s) with renv::restore(): ", paste(missing_extra, collapse = ", "), call. = FALSE)
}
extra_packages

## Execute canonical OLSE/CFA script

In [ ]:
source(file.path("scripts", "03_olse_cfa_measurement.R"), local = new.env(parent = globalenv()))

## Verify and preview notebook OLSE/CFA outputs

In [ ]:
olse_outputs <- c(
  file.path(OUT_OLSE, "olse_reliability_alpha.csv"),
  file.path(OUT_OLSE, "cfa_5factor_fit.csv"),
  file.path(OUT_OLSE, "cfa_higher_order_fit.csv"),
  file.path(OUT_OLSE, "olse_pre_post_measurement_comparability.csv"),
  file.path(OUT_OLSE, "olse_cfa_fit_comparison.csv"),
  file.path(DATA_PROCESSED, "olse_scored_for_sem.csv")
)
assert_files_exist(olse_outputs)

## Compare against script-generated OLSE/CFA outputs

In [ ]:
comparison <- write_comparison_manifest("03_olse_cfa_comparison.csv") |>
  dplyr::filter(
    grepl("^olse_measurement/", relative_path) |
      relative_path == "olse_scored_for_sem.csv" |
      relative_path == "project_soar_olse_scored_for_sem_v4.csv"
  )
comparison |>
  dplyr::count(group, status)

In [ ]:
comparison |>
  dplyr::filter(status != "match") |>
  dplyr::select(group, relative_path, status, script_path, notebook_path)

In [ ]:
preview_csv(file.path(OUT_OLSE, "olse_reliability_alpha.csv"), n = 20)

In [ ]:
preview_csv(file.path(OUT_OLSE, "olse_cfa_fit_comparison.csv"), n = 30)

In [ ]:
preview_csv(file.path(DATA_PROCESSED, "olse_scored_for_sem.csv"), n = 8)